# TAB 68B: exp68b_spectral — EXP-68B (unified latent + spectral c_B + AdaLN + normalized L1) - Colab
# Delta so với EXP-68A (và so với bản exp68 đã commit):
#   1. --unified_latent true, --latent_disentanglement false: 1 latent 64D thay 2 subspace.
#   2. --spectral_condition_token true + --spectral_phase_encoding cos_sin: token 24-D gồm
#      (w_k cos theta_k, w_k sin theta_k) và E_k/sumE, w_k = sqrt(E_k/sumE).
#      - Không còn gián đoạn tại +-pi (nhánh raw theta/pi nhảy 2.0 khi lật dấu).
#      - Bin có năng lượng ~0 (pha là nhiễu số học) bị nhân bởi w_k ~ 0 => không bơm nhiễu.
#      - Đo trên 81 file: token tách waveform (L2 2.82) và sensor (2.22) nhưng gần như mù
#        specimen (0.28) và liftoff (0.18) => đúng nghĩa "condition token".
#   3. --condition_modulation adaln + cond_in được L2-normalize trước AdaLN (trainer cũ truyền
#      thô => gamma/beta không chặn => c_B thành "bias bank" từng scan: đây là cơ chế thật của
#      triệu chứng "dùng c_B là học cực nhanh").
#   4. --cov_weight 0.2 (full-rank 64D VICReg, anti_hijack_ratio 0).
#   5. --ref_patch_mode corner_median: patch nền [10:15,10:15] của bản commit chứa 96% pixel
#      khuyết tật trên Corrosion (đo trên GT) => c_B đang là dấu vân tay khuyết tật chứ không phải
#      hàm truyền. corner_median = trung vị 4 góc: 0% pixel khuyết tật, cách khuyết tật >= 8.5 px.
# Bar: test LP phải lệch > 0.04 so exp63 (0.6435) mới được coi là có tác dụng thật.

# VẬN HÀNH (đã sửa): `runs/` được symlink sang Drive + assert kiểm chứng; cell cuối copy
# dự phòng nếu symlink hỏng. Preflight kiểm tra `train.py` có đủ flag trước khi train
# (lỗi "unrecognized arguments" của 68B là do clone Colab thiếu code mới).

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import os, glob
DATA = '/content/drive/MyDrive/PECT_Dataset/'
DRIVE_RUNS = '/content/drive/MyDrive/PECT_Dataset/runs/'
os.makedirs(DRIVE_RUNS, exist_ok=True)
# Verify: TDMS + GT masks phai thay tu Drive
tdms = [f for f in glob.glob(os.path.join(DATA, '**', '*.tdms'), recursive=True) if not f.endswith('_index.tdms')]
print('TDMS:', len(tdms))
for p in [os.path.join(DATA, 'ground_truth/corrosion/corrosion_gt_mask.npy'),
          os.path.join(DATA, 'ground_truth/rivet_v1/rivet_v1_gt_mask.npy'),
          os.path.join(DATA, 'ground_truth/rivet_v2/rivet_v2_gt_mask.npy')]:
    print(('OK  ' if os.path.exists(p) else 'MISS') + ' ' + p)

In [ ]:
!test -d GraduationProject && (cd GraduationProject && git fetch origin && git checkout Define_PECT_JEPA && git reset --hard origin/Define_PECT_JEPA) || git clone -b Define_PECT_JEPA https://github.com/TranAnh35/GraduationProject.git
!cd GraduationProject && git log --oneline -3

In [ ]:
# [FIX] sklearn version co sample_weight trong MLPClassifier.fit -> ep dung 1.7.2 (giong env lab)
!pip install -q -U "scikit-learn==1.7.2" nptdms scipy tqdm 2>&1 | tail -2
!cd GraduationProject && python -c "import torch, sklearn, scipy, nptdms; print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(), '| sklearn:', sklearn.__version__)"

In [ ]:
!mkdir -p "$DRIVE_RUNS"
!cd GraduationProject && mkdir -p experiments/cross_condition && if [ ! -L experiments/cross_condition/runs ]; then mv experiments/cross_condition/runs/* "$DRIVE_RUNS/" 2>/dev/null; rm -rf experiments/cross_condition/runs && ln -s "$DRIVE_RUNS" experiments/cross_condition/runs && echo LINKED; else echo ALREADY_LINKED; fi
!if [ -e GraduationProject/experiments/cross_condition/runs ] && [ ! -L GraduationProject/experiments/cross_condition/runs ]; then mv GraduationProject/experiments/cross_condition/runs GraduationProject/experiments/cross_condition/runs_local_bak; fi
!ln -sfn "$DRIVE_RUNS" GraduationProject/experiments/cross_condition/runs
!ls -la GraduationProject/experiments/cross_condition/ | grep runs

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

# ===== EXP-68B CONFIG: Unified 64D + spectral c_B (cos/sin) + AdaLN + normalized L1 =====
EXP_NAME = 'exp68b_spectral'

BASE_FLAGS = '--star_radii 2 4 7 --tokenizer_type energy_adaptive --latent_disentanglement false --unified_latent true --two_stream_backbone false --embed_dim 64 --var_weight 0.5 --cov_weight 0.2 --var_gamma 1.0 --scan_centered_vicreg true --inv_start_epoch 11 --inv_ramp_epochs 20 --inv_cosine true --scan_centered_inv true --inv_weight 0.2 --diverge_patience 0 --device cuda'
EXTRA = '--continuous_pos_embed true --contiguous_masking true --relative_energy_anchor true --bi_directional_relative_energy true --patch_polarity_align true --spectral_condition_token true --spectral_phase_encoding cos_sin --use_condition_token true --condition_modulation adaln --pred_loss_type normalized_l1 --anti_hijack_ratio 0.0 --self_focus_gamma 0.0 --ref_patch_mode corner_median'

os.environ.update({'EXP_NAME': EXP_NAME, 'BASE_FLAGS': BASE_FLAGS, 'EXTRA': EXTRA, 'DATA': DATA, 'DRIVE_RUNS': DRIVE_RUNS})
print('EXPERIMENT:', EXP_NAME)
print('FLAGS:', BASE_FLAGS + ' ' + EXTRA)

In [ ]:
# Cell Train (Fresh Run): exp68b_spectral — unified latent + spectral c_B + AdaLN + normalized L1
!cd GraduationProject && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.train --exp_name $EXP_NAME --add_timestamp true $BASE_FLAGS $EXTRA --data_dir "$DATA" --epochs 100 --eval_val_every 1

In [ ]:
# ==============================================================================
# Resume training tiep tuc tu DRIVE_RUNS (CHỈ chạy khi session trước đã dừng hẳn)
# ==============================================================================
import os, glob

# 1. Tim run folder trong DRIVE_RUNS (uu tien run moi nhat hoac chi dinh run cu the)
SPECIFIC_RUN = ""  # De trong de tu dong lay run moi nhat, hoac dien vi du: "exp60_contpos_asyminv_fullctx_20261017_223000"

if SPECIFIC_RUN:
    run_dir = os.path.join(DRIVE_RUNS, SPECIFIC_RUN)
else:
    candidates = sorted(glob.glob(os.path.join(DRIVE_RUNS, f"{EXP_NAME}_*")), key=os.path.getmtime)
    if not candidates:
        raise FileNotFoundError(f"Khong tim thay thu muc run nao cho {EXP_NAME} trong {DRIVE_RUNS}")
    run_dir = candidates[-1]

print(f"[*] Thu muc run tren Drive: {run_dir}")

# 2. Xac dinh checkpoint (uu tien latest de tiep tuc epoch & optimizer/scaler state, fallback sang best)
ckpt_latest = os.path.join(run_dir, "checkpoints", "latest_cross_condition_jepa.pt")
ckpt_best = os.path.join(run_dir, "checkpoints", "best_cross_condition_jepa.pt")

if os.path.exists(ckpt_latest):
    resume_ckpt = ckpt_latest
elif os.path.exists(ckpt_best):
    resume_ckpt = ckpt_best
else:
    raise FileNotFoundError(f"Khong tim thay checkpoint trong {os.path.join(run_dir, 'checkpoints')}")

print(f"[*] Checkpoint resume: {resume_ckpt} ({os.path.getsize(resume_ckpt) / 1024**2:.1f} MB)")

# 3. Kiem tra tien do da train tu metrics_epoch.csv
csv_path = os.path.join(run_dir, "evaluation_results", "metrics_epoch.csv")
if not os.path.exists(csv_path):
    csv_path = os.path.join(run_dir, "metrics_epoch.csv")
if os.path.exists(csv_path):
    with open(csv_path, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if line.strip()]
        if len(lines) > 1:
            last_line = lines[-1].split(",")
            print(f"[*] Da hoan thanh epoch: {last_line[0]} | Val AUC gan nhat: {last_line[14] if len(last_line) > 14 else 'N/A'}")

# 4. Dam bao symlink GraduationProject/experiments/cross_condition/runs -> DRIVE_RUNS
os.system(f'mkdir -p GraduationProject/experiments/cross_condition && ln -sfn "{DRIVE_RUNS}" GraduationProject/experiments/cross_condition/runs')

# 5. Export checkpoint path sang environ va thuc hien resume training
os.environ["RESUME_CKPT"] = resume_ckpt
!cd GraduationProject && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.train --exp_name "$EXP_NAME" $BASE_FLAGS $EXTRA --data_dir "$DATA" --epochs 100 --eval_val_every 1 --resume_checkpoint "$RESUME_CKPT"

In [ ]:
# ==============================================================================
# 1. Huấn luyện các mô hình Downstream (Linear/MLP Probes, Sizing, UDA Suite)
# ==============================================================================
import os, glob

run_dir = os.environ.get('LAST_RUN_DIR', None)
if not run_dir or not os.path.exists(run_dir):
    candidates = sorted(glob.glob(os.path.join(DRIVE_RUNS, f"{EXP_NAME}_*")))
    if not candidates:
        raise FileNotFoundError(f"Không tìm thấy thư mục run cho {EXP_NAME} trong {DRIVE_RUNS}")
    run_dir = candidates[-1]

ckpt_best = os.path.join(run_dir, "checkpoints", "best_cross_condition_jepa.pt")
ckpt_latest = os.path.join(run_dir, "checkpoints", "latest_cross_condition_jepa.pt")
eval_ckpt = ckpt_best if os.path.exists(ckpt_best) else ckpt_latest

if not os.path.exists(eval_ckpt):
    raise FileNotFoundError(f"Không tìm thấy checkpoint trong {os.path.join(run_dir, 'checkpoints')}")

print(f"[*] Checkpoint được chọn để train downstream: {eval_ckpt}")
os.environ["EVAL_CKPT"] = eval_ckpt

# Huấn luyện probe Detection (Linear/MLP), Sizing (Ridge/MLP) (bo qua UDA voi --skip_uda true)
# Kết quả model weights sẽ được lưu tự động vào: <run_dir>/downstream_models/
!cd GraduationProject && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.downstream.train_downstream --checkpoint "$EVAL_CKPT" --data_dir "$DATA" --device cuda --skip_uda true

In [ ]:
# ==============================================================================
# Full Downstream Evaluation + Readouts Diagnostic sau khi train xong
# ==============================================================================
import os, glob

run_dir = os.environ.get('LAST_RUN_DIR', None)
if not run_dir or not os.path.exists(run_dir):
    candidates = sorted(glob.glob(os.path.join(DRIVE_RUNS, f"{EXP_NAME}_*")))
    if not candidates:
        raise FileNotFoundError(f"Khong tim thay thu muc run nao cho {EXP_NAME} trong {DRIVE_RUNS}")
    run_dir = candidates[-1]

ckpt_best = os.path.join(run_dir, "checkpoints", "best_cross_condition_jepa.pt")
ckpt_latest = os.path.join(run_dir, "checkpoints", "latest_cross_condition_jepa.pt")

eval_ckpt = ckpt_best if os.path.exists(ckpt_best) else ckpt_latest
if not os.path.exists(eval_ckpt):
    raise FileNotFoundError(f"Khong tim thay checkpoint trong {os.path.join(run_dir, 'checkpoints')}")

print(f"[*] Dang evaluate checkpoint: {eval_ckpt}")
os.environ["EVAL_CKPT"] = eval_ckpt

!cd GraduationProject && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.evaluate --checkpoint "$EVAL_CKPT" --data_dir "$DATA" --device cuda --eval_tta true && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.diagnose_readouts --checkpoint "$EVAL_CKPT" --data_dir "$DATA" --device cuda

In [ ]:
# ==============================================================================
# BẢNG TỔNG HỢP KẾT QUẢ ĐỊNH LƯỢNG (DOWNSTREAM MASTER BENCHMARK & TTA)
# ==============================================================================
import json
import pandas as pd
from IPython.display import display, HTML

summary_path = os.path.join(eval_out_dir, 'master_benchmark_summary.json')
if os.path.exists(summary_path):
    with open(summary_path, 'r', encoding='utf-8') as f:
        summary = json.load(f)
    
    print('=' * 80)
    print(' MASTER BENCHMARK EVALUATION SUMMARY')
    print('=' * 80)
    
    # 1. Bảng theo Splits
    splits_data = []
    for sp, data in summary.get('by_split', {}).items():
        splits_data.append({
            'Split': sp.upper(),
            'Files': data.get('count', 0),
            'LP AUC': f"{data.get('linear_probe_auc_mean', 0.0):.4f} ± {data.get('linear_probe_auc_std', 0.0):.4f}",
            'LP AP': f"{data.get('linear_probe_ap_mean', 0.0):.4f}",
            'Maha AUC': f"{data.get('mahalanobis_auc_mean', 0.0):.4f}",
            'CNR': f"{data.get('cnr_mean', 0.0):.2f}",
            'Seg IoU': f"{data.get('segmentation_iou_mean', 0.0):.4f}",
            'Seg Dice': f"{data.get('segmentation_dice_mean', 0.0):.4f}",
            'Sizing R²': f"{data.get('sizing_r2_mean', 0.0):.4f}",
            'MAE (mm)': f"{data.get('sizing_mae_mean', 0.0):.3f}"
        })
    df_splits = pd.DataFrame(splits_data)
    display(HTML('<h3>1. Hiệu năng theo Tập dữ liệu (Splits)</h3>'))
    display(df_splits)
    
    # 2. Bảng theo Cảm biến (Sensor Shift)
    sensors_data = []
    for sn, data in summary.get('by_sensor', {}).items():
        sensors_data.append({
            'Sensor': sn,
            'Files': data.get('count', 0),
            'LP AUC': f"{data.get('linear_probe_auc_mean', 0.0):.4f} ± {data.get('linear_probe_auc_std', 0.0):.4f}",
            'LP AP': f"{data.get('linear_probe_ap_mean', 0.0):.4f}",
            'Maha AUC': f"{data.get('mahalanobis_auc_mean', 0.0):.4f}",
            'CNR': f"{data.get('cnr_mean', 0.0):.2f}",
            'Seg IoU': f"{data.get('segmentation_iou_mean', 0.0):.4f}"
        })
    df_sensors = pd.DataFrame(sensors_data)
    display(HTML('<h3>2. Hiệu năng theo Loại Cảm biến (Sensor)</h3>'))
    display(df_sensors)

    # 3. Bảng Canonical Test-Time Adaptation (TTA) Benchmark
    tta_res = summary.get('canonical_test_time_adaptation', {})
    if tta_res and 'shifts' in tta_res:
        tta_data = []
        for shift_axis, data in tta_res['shifts'].items():
            base = data.get('baseline_zero_shot', {})
            tta_lp = data.get('test_time_adapted_lp', {})
            tta_mh = data.get('test_time_adapted_maha', {})
            delta_lp = data.get('deltas', {}).get('delta_lp_auc', 0.0)
            delta_mh = data.get('deltas', {}).get('delta_maha_auc', 0.0)
            tta_data.append({
                'Shift Axis': shift_axis.upper(),
                'Target Files': data.get('target_count', 0),
                'Zero-Shot LP AUC': f"{base.get('mean_auc', 0.0):.4f}",
                'TTA LP AUC': f"{tta_lp.get('mean_auc', 0.0):.4f}",
                'Δ LP AUC': f"{delta_lp:+.4f}",
                'TTA Maha AUC': f"{tta_mh.get('mean_auc', 0.0):.4f}",
                'Δ Maha AUC': f"{delta_mh:+.4f}",
                'TTA LP AP': f"{tta_lp.get('mean_ap', 0.0):.4f}"
            })
        df_tta = pd.DataFrame(tta_data)
        display(HTML('<h3>3. Kết quả Canonical Test-Time Adaptation (TTA) Benchmark</h3>'))
        display(df_tta)
else:
    print(f'[!] Chưa tìm thấy {summary_path}')


In [ ]:
# ==============================================================================
# HIỂN THỊ TRỰC QUAN C-SCAN ANOMALY DETECTION & TTA MAPS
# ==============================================================================
import glob
from PIL import Image
import matplotlib.pyplot as plt

# 1. Hiển thị Anomaly Detection (4-panel subplots: CAD GT, Raw Vpp, Mahalanobis, Linear Probe)
anom_plots = sorted(glob.glob(os.path.join(eval_out_dir, 'task_1_anomaly_detection', '**', '*.png'), recursive=True))
print(f'[*] Tìm thấy {len(anom_plots)} biểu đồ Anomaly Detection.')

if anom_plots:
    for p in anom_plots[:3]:
        img = Image.open(p)
        plt.figure(figsize=(18, 5))
        plt.imshow(img)
        plt.axis('off')
        plt.title(f'Anomaly Detection 4-Panel Map: {os.path.basename(p)}', fontsize=12, fontweight='bold')
        plt.show()

# 2. Hiển thị TTA Comparison (5-panel subplots: CAD GT, Raw Vpp, Zero-Shot LP, TTA LP, TTA Maha)
tta_plots = sorted(glob.glob(os.path.join(eval_out_dir, 'task_4_canonical_tta', '*.png')))
print(f'[*] Tìm thấy {len(tta_plots)} biểu đồ TTA Benchmark.')

if tta_plots:
    for p in tta_plots[:3]:
        img = Image.open(p)
        plt.figure(figsize=(22, 5))
        plt.imshow(img)
        plt.axis('off')
        plt.title(f'TTA Adaptation 5-Panel Map: {os.path.basename(p)}', fontsize=12, fontweight='bold')
        plt.show()

# 3. Hiển thị Sizing Calibration Scatter Plot
sizing_plots = sorted(glob.glob(os.path.join(eval_out_dir, 'task_2_sizing_hurdle', '**', '*.png'), recursive=True))
if sizing_plots:
    img = Image.open(sizing_plots[0])
    plt.figure(figsize=(8, 7))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f'Sizing Hurdle Calibration Scatter: {os.path.basename(sizing_plots[0])}', fontsize=12, fontweight='bold')
    plt.show()


In [ ]:
# ==============================================================================
# KIỂM TRA ĐỒNG BỘ TOÀN VẸN TRÊN GOOGLE DRIVE
# ==============================================================================
print('=' * 80)
print(f'Kiểm tra thư mục kết quả: {run_dir}')
for root, dirs, files in os.walk(run_dir):
    level = root.replace(run_dir, '').count(os.sep)
    indent = ' ' * 4 * level
    print(f'{indent}{os.path.basename(root)}/ ({len(files)} files)')
    if level <= 2:
        for f in files[:5]:
            fpath = os.path.join(root, f)
            sz = os.path.getsize(fpath) / (1024 * 1024)
            print(f'{indent}    - {f} ({sz:.2f} MB)')
        if len(files) > 5:
            print(f'{indent}    ... và {len(files) - 5} files khác')
print('=' * 80)
print('[OK] Toàn bộ dữ liệu huấn luyện, weights và kết quả đánh giá đã được đồng bộ an toàn trên Google Drive!')
